# Projects in data science: Linear & Logistic Regression

### We will be using lots of Scikit-Learn throughout this exercise so look up their documentation if get stuck along the way :-).

In [3]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns

## Exercise 1: Linear Regression

A housing company reached out to you. They want you to help them predict the price of apartments based on their size (square meters) and average number of sun hours per day. They provide you with a dataset of 150 apartments, with the following columns:
- `size_m2`: size of the apartment in square meters
- `sun_hours`: average number of sun hours per day
- `price_eur`: price of the apartment in euros

However, somehow there are some outlier due to typos in the dataset. Your tasks are:
1. Inspect the dataset and identify any outliers. Remove them from the dataset.
2. Split the dataset into train and test sets.
3. Train a linear regression model on the train set.
4. Evaluate the model on the test set using MSE and MAE.

The following cell generates the dataset for you. Run it but don't look at the code too closely yet, as it might reveal some of the outliers.

In [4]:
# set random seed for reproducibility
np.random.seed(42)

# generate synthetic data
n_samples = 150
size_m2 = np.random.uniform(30, 200, n_samples).round(0)   # house size in square meters
sun_hours = np.random.uniform(2, 10, n_samples).round(1)   # avg sun hours per day

price = (
1500 * size_m2 # 1.500€ per square meter
+ 3000 * sun_hours # sunnier = more expensive
+ 50000 # base price
+ np.random.normal(0, 15000, n_samples) # noise
)

# add some outliers to the sun hours
outlier_idx_sun = np.random.choice(n_samples, size=5, replace=False)
sun_hours[outlier_idx_sun] = np.random.uniform(50, 230, size=5).round(1)

# add some outliers to the size
outlier_idx_size = np.random.choice(n_samples, size=2, replace=False)
size_m2[outlier_idx_size] = 0

### 1.1 Inspecting the dataset and identifying outliers

In [17]:
# create dataframe
df = pd.DataFrame({
    'size_m2': size_m2,
    'sun_hours': sun_hours,
    'price_eur': price.round(0)
})

# ------------- your code below -------------

# You can test for outliars, min, max, etc.

In [18]:
# If you have found any anomalies that doesn't realisticly fit into the dataset, you can remove them.

### 1.2 Split the dataset into training and test sets

In [19]:
# Tip: You can use train_test_split() from sklearn to split the dataset. Also remember to add a random state for reproducibility.

# ------------- your code below -------------

### 1.3 Train a linear regression model on the training set

In [20]:
# Tip: Again, have a look at the documentation of sklearn for LinearRegression

# ------------- your code below -------------
# 1. train the model

# 2. print the coefficients to see how the model performed

### 1.4 Evaluate the model on the test set using MSE and MAE

In [9]:
# Tip: Have a look at the documentation of sklearn for mean_squared_error() and mean_absolute_error()

# ------------- your code below -------------
# 1. make predictions on the test set
y_test_pred = model.predict(X_test)

# 2. evaluate the model
print("Test performance:")
print(f"MSE: {np.sqrt(mean_squared_error(y_test, y_test_pred))}")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred)}")

Test performance:
MSE: 13702.048142710606
MAE: 10529.70839472305


### Interactive 3D visualization of the fitted hyperplane
Don't worry too much about the code below. However, you might have different variable names in your code, so make sure to adjust the code below accordingly or go back and change your variables to make the code below work.

In [10]:
# create a grid over the feature space to plot the hyperplane
size_range = np.linspace(df['size_m2'].min(), df['size_m2'].max(), 30)
sun_range = np.linspace(df['sun_hours'].min(), df['sun_hours'].max(), 30)
size_grid, sun_grid = np.meshgrid(size_range, sun_range)

# compute hyperplane directly from learned coefficients
# price = intercept + coef[0]*size + coef[1]*sun_hours
price_grid = (
    model.intercept_
    + model.coef_[0] * size_grid
    + model.coef_[1] * sun_grid
)

fig = go.Figure()

# training points
fig.add_trace(go.Scatter3d(
    x=X_train['size_m2'], y=X_train['sun_hours'], z=y_train,
    mode='markers',
    marker=dict(size=4, color='steelblue', opacity=0.7),
    name='Train'
))

# test points
fig.add_trace(go.Scatter3d(
    x=X_test['size_m2'], y=X_test['sun_hours'], z=y_test,
    mode='markers',
    marker=dict(size=4, color='orange', opacity=0.7),
    name='Test'
))

# regression hyperplane
fig.add_trace(go.Surface(
    x=size_range, y=sun_range, z=price_grid,
    opacity=0.5,
    colorscale='Greens',
    showscale=False,
))

fig.update_layout(
    title='Your model',
    scene=dict(
        xaxis_title='Size (square meters)',
        yaxis_title='Sun hours/day',
        zaxis_title='Price (€)',
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
    scene_camera=dict(
        eye=dict(x=-1.5, y=-2, z=0.8)
    )
)

fig.show()


## Exercise 2: Logistic Regression

Again, your help is needed! A hospital has only one CT scanner (3D X-ray machine) available and they want to use it to help patients with colon cancer. However, there are too many people complaining about stomach pain and they don't know who to prioritize for the CT scan. Most of them have a stomach ache for a few days and then it goes away, but some of them have colon cancer and they need to be prioritized for the CT scan. The hospital has collected data from 500 patients, with the following columns:
- `age`: age of the patient (colon cancer is more common in older people)
- `stomach_pain_duration`: duration of the stomach pain in days (more days of stomach pain might indicate a higher risk of colon cancer)
- `has_colon_cancer`: whether the patient has colon cancer (1) or not (0)

Your tasks are:
1. Think about the objective of this task. Do we want to correctly classify patients overall or do we want to prioritize identifying of a certain class? How does that effect the choice of evaluation metric?
2. Create a train, validation and test split of the dataset.
3. Train a logistic regression model on the dataset. Make sure to handle the class imbalance in the dataset.
4. Evaluate your best model on the test set. Plot the confusion matrix for the test set.
5. Reflect on the results. Did you achieve the objective you set in the first step? If not, what could you do to improve the model?

The following cell generates the dataset for you. You just need to run it.

In [11]:
# generate synthetic dataset for logistic regression
np.random.seed(42)
n_patients = 500

age = np.random.randint(30, 85, n_patients) # age in years
stomach_pain_duration = np.random.randint(1, 30, n_patients) # days of stomach pain

log_odds = (
-9.0
+ 0.1 * age
+ 0.08 * stomach_pain_duration
)
prob_cancer = 1 / (1 + np.exp(-log_odds)) # sigmoid
has_colon_cancer = np.random.binomial(1, prob_cancer)

df_cancer = pd.DataFrame({
'age': age,
'stomach_pain_duration': stomach_pain_duration,
'has_colon_cancer': has_colon_cancer
})

print(f"Patients: {n_patients}")
print(f"With colon cancer: {has_colon_cancer.sum()} ({has_colon_cancer.mean()*100:.1f}%)")
print(f"Without colon cancer: {(1 - has_colon_cancer).sum()} ({(1 - has_colon_cancer).mean()*100:.1f}%)")
df_cancer.head()

Patients: 500
With colon cancer: 102 (20.4%)
Without colon cancer: 398 (79.6%)


,age,stomach_pain_duration,has_colon_cancer
0,68,3,0
1,81,23,0
2,58,8,0
3,44,20,0
4,72,16,1


### Let's have a look at the dataset first

In [12]:
fig = px.scatter(
df_cancer,
x='age',
y='stomach_pain_duration',
color=df_cancer['has_colon_cancer'].map({0: 'No cancer', 1: 'Colon cancer'}),
color_discrete_map={'No cancer': 'steelblue', 'Colon cancer': 'crimson'},
labels={
'age': 'Age (years)',
'stomach_pain_duration': 'Stomach pain duration (days)',
'color': 'Diagnosis'
},
title='Patient dataset',
opacity=0.7,
)

fig.update_traces(marker=dict(size=8))
fig.show()

### 2.1 Think about the objective of this task. Do we want to correctly classify patients overall or do we want to prioritize identifying of a certain class? How does that effect the choice of evaluation metric?



**Write your thoughts here:**

### 2.2 Create a train, validation and test split of the dataset.

In [21]:
# Tips:
# As before, you can use train_test_split() from sklearn to split the dataset. 
# Remember to stratify the split to maintain the same class distribution in each split. 
# Also, remember to add a random state for reproducibility.

# ------------- your code below -------------

# split: 60% train, 20% validation, 20% test

# print the number of samples and class distribution in each split

### 2.3 Train a logistic regression model on the dataset. Make sure to handle the class imbalance in the dataset.

In [ ]:
# Tips:
# The features are on different scales. See documentation for StandardScaler() to scale the features before training the model.
# Have a look at the class_weight parameter to handle class imbalance in the dataset.
# You can use accuracy_score() and roc_auc_score() from sklearn. See documentation.

# ------------- your code below -------------

# 1. scale features
# it is important to fit the scaler only on the training data and then apply the same transformation to the validation and test data

# 2. train logistic regression model.

# 3. optimize you model on the validation set by trying out different class_weight values. 
# (You can do that manually by changing the class_weight parameter and re-running the code)

# get prediction probabilities for the positive class (cancer) on the validation set


# 4. print the accuracy and AUC for the validation set.

Validation performance (threshold = 0.5):
Accuracy: 0.720
AUC: 0.823


### 2.4 Evaluate your "best" model on the test set. Plot the confusion matrix for the test set.

In [22]:
# ------------- your code below -------------
# 1. make predictions on the test set

# 2. print the accuracy and AUC for the test set.

In [23]:
# Have a look at confusion_matrix() from sklearn to plot the confusion matrix

# ------------- your code below -------------
# plot confusion matrix for the test set

### 2.5 Reflect on the results. Did you achieve the objective you set in the first step? If not, what could you do to improve the model?

**Write your reflection here:**